In [1]:
suppressPackageStartupMessages(library(ArchR))
suppressPackageStartupMessages(library(parallel))



                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .______      
          /   \     |   _ 

In [2]:
here::i_am("atac/chrombpnet/01_export_fragments.ipynb")

source(here::here("settings.R"))
source(here::here("utils.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/10_Eomes_invitro_gut/code



In [6]:
# outdir
args = list()
args$metadata = file.path(io$basedir, 'results/rna_atac/clustering/metadata_mofa_clusters.txt.gz')
args$early_annotations =  file.path(io$basedir, 'results/rna_atac/clustering/early_celltypes.txt.gz')

args$outdir = file.path(io$basedir, 'results/atac/chrombpnet/')
dir.create(args$outdir, recursive=TRUE, showWarnings =FALSE)

In [11]:
# Load meta
meta = fread(args$metadata)[day=='D3' & genotype == 'WT']

early_annotations = fread(args$early_annotations)

meta$celltype_manual = early_annotations[match(meta$cell, cell), celltype_manual]
meta$celltype_manual2 = ifelse(meta$celltype_manual == 'PS Eomes-dependent', 'PS_EOd',
                               ifelse(meta$celltype_manual %in% paste0('PS ', 1:3), 'PS_EOi', 'other'))
                               

In [12]:
head(meta)

cell,barcode,sample,nFeature_RNA,nCount_RNA,mitochondrial_percent_RNA,ribosomal_percent_RNA,alias,day,genotype,⋯,TSSEnrichment_atac,ReadsInTSS_atac,PromoterRatio_atac,NucleosomeRatio_atac,nFrags_atac,BlacklistRatio_atac,pass_atacQC,mofa_cluster,celltype_manual,celltype_manual2
<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<chr>,<chr>,<chr>,⋯,<dbl>,<int>,<dbl>,<dbl>,<int>,<dbl>,<lgl>,<int>,<chr>,<chr>
RBG43475#AAACATGCAGGATAAC-1,AAACATGCAGGATAAC-1,RBG43475,3698,10514,21.44,6.39,D3_WT_rep1,D3,WT,⋯,11.69,496,0.19,1.22,5172,0.01,TRUE,3,PS 3,PS_EOi
RBG43475#AAACATGCAGTAAGTA-1,AAACATGCAGTAAGTA-1,RBG43475,4072,12739,17.43,4.74,D3_WT_rep1,D3,WT,⋯,12.41,4576,0.19,1.48,47540,0.02,TRUE,17,PS 3,PS_EOi
RBG43475#AAACCAACACCAGCAT-1,AAACCAACACCAGCAT-1,RBG43475,3981,12161,20.60,7.99,D3_WT_rep1,D3,WT,⋯,17.98,4268,0.29,0.92,26297,0.03,TRUE,7,PS 1,PS_EOi
RBG43475#AAACCAACAGAGAGCC-1,AAACCAACAGAGAGCC-1,RBG43475,4253,11827,12.77,6.99,D3_WT_rep1,D3,WT,⋯,9.29,3578,0.15,1.40,51366,0.02,TRUE,19,PS Eomes-dependent,PS_EOd
RBG43475#AAACCGAAGCTTAACA-1,AAACCGAAGCTTAACA-1,RBG43475,4529,15998,16.04,4.38,D3_WT_rep1,D3,WT,⋯,13.95,5192,0.21,1.36,46712,0.02,TRUE,17,PS Eomes-dependent,PS_EOd
RBG43475#AAACCGAAGTTAGACC-1,AAACCGAAGTTAGACC-1,RBG43475,3465,8540,13.13,8.09,D3_WT_rep1,D3,WT,⋯,14.14,1614,0.19,0.94,15419,0.02,TRUE,19,PS 2,PS_EOi


In [13]:
unique(meta$sample)

[1] "RBG43475" "RBG43476"

In [19]:
chrs = c('chr1','chr10','chr11','chr12','chr13','chr14','chr15','chr16','chr17','chr18','chr19','chr2','chr3','chr4','chr5','chr6','chr7','chr8','chr9', 'chrX')
# exclude samples
samples_incl = unique(meta$sample)

# Accidentally doing it for all cell types --> run ChromBPNet only on those of interest
celltypes_in = c(
'PS_EOi', 'PS_EOd'
)


# Export fragments of celltype cells in subset of samples
mclapply(samples_incl, function(x){
    # List cells per sample
    message(x)
    # filter right samples
    tmp_meta = meta[sample == x]  
    # Keep relevant cell types
    celltypes = tmp_meta %>%
        .[, .N, by = 'celltype_manual2'] %>% 
        unique(by = 'celltype_manual2') %>%
        .[celltype_manual2 %in% celltypes_in] %>%
        .[N > 20] %>% 
        .$celltype_manual2
    message(celltypes)
    
    # Load fragment file
    file = sprintf('/rds/project/rds-SDzz0CATGms/users/bt392/10_Eomes_invitro_gut/original/%s/outs/atac_fragments.tsv.gz', x)
    fragment = suppressWarnings(fread(file,
                                        tmpdir = '/rds/project/rds-SDzz0CATGms/users/bt392/software/tmp'))
    message('before filtering:')
    message(nrow(fragment))
    
    mclapply(celltypes, function(i){
        message(i)
        # Only run if file doesn't exist yet
        if(!file.exists(sprintf('%s/fragments/%s_%s.tsv.gz', args$outdir, i, x))){
            # determine cells
            tmp_cell = tmp_meta[celltype_manual2 == i, barcode]
            
            # Filter right cells & chrs from fragment file
            fragment = fragment %>% 
                setnames(c('chr', 'start', 'end', 'cell', 'reads')) %>%
                # .[chr == 'chr1'] %>%
                .[cell %in% tmp_cell] %>% 
                .[chr %in% chrs]

            message('after filtering:')
            message(nrow(fragment))

            fwrite(fragment, sprintf('%s/fragments/%s_%s.tsv.gz', args$outdir, i, x), sep = '\t')
        }
    }, mc.cores = 1)
}, mc.cores = 1)

RBG43475

PS_EOiPS_EOd

before filtering:

212695009

PS_EOi

after filtering:

102416736

PS_EOd

after filtering:

48959931

RBG43476

PS_EOiPS_EOd

before filtering:

198058457

PS_EOi

after filtering:

96820093

PS_EOd

after filtering:

39750342



[[1]]
[[1]][[1]]
NULL

[[1]][[2]]
NULL


[[2]]
[[2]][[1]]
NULL

[[2]][[2]]
NULL

In [20]:
celltypes_in = c(
'PS_EOi', 'PS_EOd'
)


lapply(celltypes_in, function(i){    
    message(i)
    
    # List files per cell type
    files = list.files(file.path(args$outdir, 'fragments/'), pattern = i)
    # overwrite combined fragment file if it exists
    if(length(grep('fragment', files))>0){
        files = files[-grep('fragment', files)] 
    }
    message(files)
    # Load in the separate fragment files per cell type
    tmp = mclapply(files, function(x){
        tmp_frag = fread(sprintf('%s/fragments/%s', args$outdir, x), sep = '\t',
                                        tmpdir = '/rds/project/rds-SDzz0CATGms/users/bt392/software/tmp')
        return(tmp_frag)
    }, mc.cores = 8) %>% rbindlist() %>% 
    .[order(chr, start)] %>% 
    .[,reads := 1]

    fwrite(tmp, sprintf('%s/fragments/%s_fragments.tsv.gz', args$outdir, i), col.names = F, sep = '\t')
})

PS_EOi

PS_EOi_RBG43475.tsv.gzPS_EOi_RBG43476.tsv.gz

PS_EOd

PS_EOd_RBG43475.tsv.gzPS_EOd_RBG43476.tsv.gz



[[1]]
NULL

[[2]]
NULL

In [27]:
ArchRProject = loadArchRProject(io$archR.directory)

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [25]:
peaks = fread(file.path(io$basedir, 'results/rna_atac/clustering/early_peaks.bed')) %>%
    setnames(c('seqnames', 'start', 'end')) %>%
    .[, middle := start + (end - start) / 2] %>%
    .[,`:=`(start = middle - 1057, 
            end = middle + 1057,
            column4 = '.',
            column5 = '.',
            column6 = '.',
            column7 = '.',
            column8 = '.',
            column9 = '.',
            column10 = 1057)] %>% 
    .[,.(seqnames, 
         start, 
         end, 
         column4,
         column5,
         column6,
         column7,
         column8,
         column9,
         column10)] %>%
    .[seqnames != 'chrX']

nrow(peaks)
head(peaks)

[1] 207385

seqnames,start,end,column4,column5,column6,column7,column8,column9,column10
<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
chr1,3034849,3036963,.,.,.,.,.,.,1057
chr1,3061931,3064045,.,.,.,.,.,.,1057
chr1,3190813,3192927,.,.,.,.,.,.,1057
chr1,3263144,3265258,.,.,.,.,.,.,1057
chr1,3324379,3326493,.,.,.,.,.,.,1057
chr1,3339830,3341944,.,.,.,.,.,.,1057


In [28]:
peaks.gr = GenomicRanges::makeGRangesFromDataFrame(peaks, keep.extra.columns = T)
overlaps = GenomicRanges::findOverlaps(peaks.gr, ArchRProject@genomeAnnotation$blacklist)

peaks.gr = peaks.gr[-queryHits(overlaps)]

In [30]:
peaks = as.data.table(peaks.gr)[,`:=`(width=NULL, strand=NULL)]
fwrite(peaks, sprintf('%s/all_peaks.bed', args$outdir), col.names = F, sep = '\t')

In [31]:
nrow(peaks)
tail(peaks)

[1] 206925

seqnames,start,end,column4,column5,column6,column7,column8,column9,column10
<fct>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
chr19,61268330,61270444,.,.,.,.,.,.,1057
chr19,61284175,61286289,.,.,.,.,.,.,1057
chr19,61307664,61309778,.,.,.,.,.,.,1057
chr19,61312469,61314583,.,.,.,.,.,.,1057
chr19,61313510,61315624,.,.,.,.,.,.,1057
chr19,61320010,61322124,.,.,.,.,.,.,1057


In [32]:
chromSizes = as.data.table(ArchRProject@genomeAnnotation$chromSizes) %>%
    .[,.(seqnames, end)]
fwrite(chromSizes, sprintf('%s/mm10.chrom.sizes', args$outdir), col.names = F, sep = '\t')

In [33]:
fwrite(as.data.table(ArchRProject@genomeAnnotation$blacklist) %>%
           .[,.(seqnames, start, end)],
       sprintf('%s/blacklist.bed.gz', args$outdir), col.names = F, sep = '\t')